# E1：从深度图片走进三维世界

这一份共同 Notebook 不训练大模型。我们先把相机、坐标变换、BEV 与 Occupancy 算对。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
from hwm.foundations import depth_to_points, points_to_occupancy
from hwm.spatial import make_camera_transform, transform_points


## 1. 深度像素怎样变成三维点

内参 `fx, fy, cx, cy` 描述相机怎样成像。已知每个像素深度，就能把它反投影到相机坐标。

In [ ]:
depth = np.full((6, 6), 4.0, dtype=np.float32)
depth[2:4, 2:4] = 2.0
points_camera = depth_to_points(depth, fx=6, fy=6, cx=2.5, cy=2.5)
print('points:', points_camera.shape, 'near/far z:', points_camera[:, 2].min(), points_camera[:, 2].max())
assert points_camera.shape == (36, 3)


## 2. 外参把不同相机放进同一世界

相机向右移动一米，同一个相机坐标点在世界坐标中也整体平移。坐标系方向或矩阵乘法写反，会让多视角无法对齐。

In [ ]:
camera_to_world = make_camera_transform(tx=1.0, yaw=0.0)
points_world = transform_points(points_camera, camera_to_world)
shift = points_world.mean(0) - points_camera.mean(0)
print('mean shift:', np.round(shift, 3))
assert np.allclose(shift, [1, 0, 0], atol=1e-5)


## 3. 点落到俯视 Occupancy

Occupancy 不关心表面颜色，只记录空间哪里已有物体。它很适合碰撞检查与驾驶未来预测。

In [ ]:
occupancy = points_to_occupancy(points_world, x_range=(-2, 4), z_range=(0, 6), resolution=0.5)
print('occupancy shape/occupied:', occupancy.shape, int(occupancy.sum()))
assert occupancy.sum() > 0


## 4. 标定误差会怎样

把平移错写成 1.3 米，整个点云都会偏移。神经网络可能在训练集上适应固定偏差，却不能让错误几何变正确。

In [ ]:
wrong_world = transform_points(points_camera, make_camera_transform(tx=1.3))
calibration_error = np.linalg.norm(wrong_world - points_world, axis=1).mean()
print('mean calibration error:', round(float(calibration_error), 3), 'm')
assert np.isclose(calibration_error, 0.3, atol=1e-5)


## 小结

相机内参负责从像素到射线，外参负责从相机到世界，BEV/Occupancy 把三维点变成可规划网格。完成 E1 后，在 E2a 神经场和 E2b 未来占用中选择一份。